In [ ]:
# Same pipeline, no PyTorch. [Mirrors Lesson 02 Embeddings]
# Here we use the ONNX-based embedder to generate the same kind of vectors
# as before, but with a much lighter runtime setup.
# The model itself can be changed by passing a different path to Embedder(),

from embedder import Embedder

embed = Embedder() # Here we could use a different model, e.g., Embedder("models/Xenova/bge-base-en-v1.5").

q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

# Encode the two queries and the document.
v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)

# Compare the similarity scores.
# The course-registration question should be more similar to the document
# than the Docker question.
score_1 = v1.dot(dv)
score_2 = v2.dot(dv)

print(f"Similarity(q1, d): {score_1}")
print(f"Similarity(q2, d): {score_2}")
print("Expected: q1 should score higher than q2 because it matches the document meaning more closely.")

Similarity(q1, d): 0.3233238799303238
Similarity(q2, d): 0.019730422395141473
Expected: q1 should score higher than q2 because it matches the document meaning more closely.


In [3]:
# Load the FAQ documents. [Mirrors Lesson 03 Embeddigngs Dataset]
# Each document contains fields such as question and answer.

from ingest import load_faq_data

documents = load_faq_data()

print(f"Loaded {len(documents)} FAQ documents")

Loaded 1350 FAQ documents


In [4]:
# Combine each question and answer into a single text string.
# This gives us one text per document to embed for retrieval.

texts = [doc["question"] + " " + doc["answer"] for doc in documents]

print(f"Prepared {len(texts)} text entries for embedding")

Prepared 1350 text entries for embedding


In [5]:
# Embed the FAQ texts in batches.
# Batch processing is more efficient than encoding one document at a time.
# We store all document vectors in X.

from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

print(f"Created embedding matrix with shape: {X.shape}")

  0%|          | 0/27 [00:00<?, ?it/s]

Created embedding matrix with shape: (1350, 384)


In [6]:
# Encode a query and compare it against all document embeddings. [Mirrors Lesson 04 Vector Search]
# Dot product works here because the vectors are normalized.
# The highest score should point to the most semantically similar FAQ entry.

query = "Can I still join the course after the start date?"
v_query = embed.encode(query)

scores = X.dot(v_query)
idx = np.argmax(scores)

print(f"Query encoded successfully with shape: {v_query.shape}")
print(f"Best matching document index: {idx}")

Query encoded successfully with shape: (384,)
Best matching document index: 2


In [7]:
# Inspect the top matching document.
# This should return the FAQ entry that best matches the query meaning.

documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}